# STORM: Structural & Tandem-Repeat Optimized Regression Models

This notebook demonstrates end-to-end usage of STORM for association testing of structural variants and tandem repeats.

In [2]:
# Install dependencies if needed
# !pip install polars pyarrow

## 1. Import STORM

In [3]:
import sys
import polars as pl
from pathlib import Path

# Import STORM Python package
import storm

# Check version and bindings
print(f"STORM version: {storm.version()}")
print(f"Rust bindings available: {storm.HAS_RUST}")

AttributeError: module 'storm' has no attribute 'version'

In [4]:
# Check fixtures directory
import os
fixtures_path = Path("../fixtures")
print("Available fixtures:")
for f in sorted(fixtures_path.glob("*")):
    print(f"  {f.name}: {f.stat().st_size} bytes")

<module 'storm' from '/Users/kiran/repositories/storm/venv/lib/python3.13/site-packages/storm/__init__.py'>

## 2. Build Cache from Input Files

STORM builds a cache from:
- Integrated SV VCF (structural variants)
- TRGT VCF (tandem repeat genotypes) 
- TRExplorer BED/JSON catalog

In [ ]:
# Build cache from fixtures using Python API
cache = storm.StormCache.build(
    sv_vcf="../fixtures/sv_small.vcf",
    trgt_vcf="../fixtures/trgt_small.vcf",
    catalog_bed="../fixtures/trexplorer.bed",
    catalog_json="../fixtures/trexplorer.json",
    output_dir="demo_cache"
)

print(f"Cache built at: {cache.cache_dir}")
print(f"Build stats: {cache._build_stats}")

## 3. Load Cache via Polars

The cache stores data in Parquet format, which Polars can load efficiently.

In [ ]:
# Verify the cache
result = storm.verify_cache("demo_cache")
print(f"Cache valid: {result['is_valid']}")
print(f"Test units: {result['num_test_units']}")
print(f"Genotypes: {result['num_genotypes']}")
print(f"Features: {result['num_features']}")
print()

# Load cache tables via StormCache properties
print("Test Units:")
print(cache.test_units)

print("\nGenotypes:")
print(cache.genotypes)

print("\nCatalog:")
print(cache.catalog)

print("\nFeatures:")
print(cache.features.head())

## 4. Run StormGLM Association Testing

STORM supports multiple models:
- Linear regression for quantitative traits
- Logistic regression for binary traits
- BinomiRare for rare variant burden testing
- Firth regression for rare events

In [ ]:
# Example: Analyzing features for association testing
# The features table contains computed values for each genotype

# Group features by unit to see summary statistics
unit_summary = features.group_by("unit_id").agg([
    pl.count().alias("n_samples"),
    pl.col("binary").sum().alias("n_carriers"),
    pl.col("S").mean().alias("mean_S"),
    pl.col("M").mean().alias("mean_M"),
    pl.col("D").mean().alias("mean_D"),
])
print("Unit Summary Statistics:")
print(unit_summary)

# In a real analysis, you would:
# 1. Load phenotype data
# 2. Join with features on sample_id
# 3. Run regression using the plan configuration
print("\nGLM analysis would be run using storm.run_glm() with Python bindings")

## 5. Explain Genotypes

Get detailed information about resolved genotypes at a locus.

In [ ]:
# Explain a specific test unit using the CLI
# First, let's see what units we have
print("Available test units:")
print(test_units["id"].to_list())

# Explain a specific unit (using the first SV unit)
unit_id = test_units["id"][0]
print(f"\n=== Explaining {unit_id} ===\n")

result = subprocess.run([
    "cargo", "run", "--", 
    "explain", unit_id,
    "--cache-dir", "notebooks/demo_cache"
], capture_output=True, text=True, cwd="..")
print(result.stdout)

# Explain for a specific sample
print(f"\n=== Explaining {unit_id} for SAMPLE1 ===\n")
result = subprocess.run([
    "cargo", "run", "--", 
    "explain", unit_id,
    "--cache-dir", "notebooks/demo_cache",
    "--sample", "SAMPLE1"
], capture_output=True, text=True, cwd="..")
print(result.stdout)

## 6. Encodings

STORM supports multiple genotype encodings:
- **S**: Sum of allele lengths (L1 + L2)
- **M**: Maximum allele length
- **D**: Absolute difference |L1 - L2|
- **Tail**: Indicator for exceeding threshold
- **Categorical**: Binned categories

In [ ]:
# Encodings are selected via the plan YAML
plan_example = """
name: my_analysis
default_encoding: s
default_model: linear

rules:
  - name: rare_binary
    condition:
      max_carriers: 10
    encoding: binary
    model: binomirare
    
  - name: cag_repeats  
    condition:
      motif_pattern: CAG
    encoding: m
    model: logistic

covariates:
  - age
  - sex
  
num_pcs: 10
"""

print("Example Plan YAML:")
print(plan_example)

## 7. Results Analysis

Results are returned as Polars DataFrames for easy analysis.

In [ ]:
# Example results analysis
# significant = results.filter(pl.col("p_value") < 5e-8)
# print(f"Genome-wide significant hits: {len(significant)}")

# by_model = results.group_by("model").agg(pl.count())
# print("\nTests by model:")
# print(by_model)

print("Results analysis demonstrated above (commented for demo)")

## Summary

STORM provides a complete workflow for:

1. **Parsing** - SV VCF, TRGT VCF, TRExplorer catalogs
2. **Resolution** - Mapping SVs to repeat loci, reconstructing diploid lengths
3. **Caching** - Arrow/Parquet storage for efficient access
4. **Analysis** - Multiple regression models and encodings
5. **Output** - Parquet results with full metadata

The Rust core ensures correctness and performance, while the Python API provides interactive exploration via Polars.